In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, Markdown, display


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'codigo' / 'pipeline_filmes.py').is_file():
            return candidate
    raise FileNotFoundError('Nao foi possivel localizar a raiz do projeto.')


PROJECT_ROOT = resolve_project_root()
CODIGO_DIR = PROJECT_ROOT / 'codigo'
if str(CODIGO_DIR) not in sys.path:
    sys.path.insert(0, str(CODIGO_DIR))

from pipeline_filmes import main

pd.set_option('display.max_columns', 30)
results = main(data_root=PROJECT_ROOT)


In [ ]:
source_summary = pd.DataFrame(results['source_summary']['datasets'])
stage_summary = pd.DataFrame(
    [
        {'etapa': etapa, **dados}
        for etapa, dados in results['stage_summary'].items()
    ]
)
output_summary = pd.DataFrame(
    [
        {'artefato': nome, 'caminho': caminho}
        for nome, caminho in results['output_summary']['arquivos_saida'].items()
    ]
)

display(Markdown('### Fontes lidas'))
display(source_summary)
display(Markdown('### Etapas do pipeline'))
display(stage_summary)
display(Markdown('### Artefatos publicados'))
display(output_summary.head(12))


In [ ]:
results['gold_df'].head(10)


In [ ]:
table_previews = [
    'receita_por_genero',
    'top_diretores',
    'filmes_por_decada',
    'top_palavras_chave',
    'correlacoes_sucesso',
    'top_filmes_receita',
    'top_filmes_roi',
]

for name in table_previews:
    display(Markdown(f'### {name}'))
    display(results['tables'][name].head(10))


In [ ]:
highlights = {
    'genero_lider_receita': results['tables']['receita_por_genero'].iloc[0]['primary_genre'],
    'diretor_lider_receita': results['tables']['top_diretores'].iloc[0]['director'],
    'decada_com_mais_filmes': int(results['tables']['filmes_por_decada'].sort_values('qtd_filmes', ascending=False).iloc[0]['decade']),
}

pd.DataFrame([highlights])


In [ ]:
for name, path in results['plot_paths'].items():
    display(Markdown(f'### {name}'))
    display(Image(filename=str(path)))
